In [ ]:
"""
langchain: Framework para construir aplicações que utilizam modelos de linguagem (LLMs), facilitando a criação de fluxos complexos.

langchain_community: Extensão do LangChain que adiciona suporte a novos conectores e ferramentas da comunidade.

langchain-huggingface: Integração do LangChain com modelos da Hugging Face, permitindo uso simplificado de LLMs e embeddings.

langchainhub: Repositório de templates de prompts e componentes reutilizáveis para LangChain.

langchain_chroma: Integração do LangChain com ChromaDB, um banco de dados vetorial para armazenar e recuperar embeddings.

"""

'\nlangchain: Framework para construir aplicações que utilizam modelos de linguagem (LLMs), facilitando a criação de fluxos complexos.\n\nlangchain_community: Extensão do LangChain que adiciona suporte a novos conectores e ferramentas da comunidade.\n\nlangchain-huggingface: Integração do LangChain com modelos da Hugging Face, permitindo uso simplificado de LLMs e embeddings.\n\nlangchainhub: Repositório de templates de prompts e componentes reutilizáveis para LangChain.\n\nlangchain_chroma: Integração do LangChain com ChromaDB, um banco de dados vetorial para armazenar e recuperar embeddings.\n\n'

In [ ]:
"""
transformers: Biblioteca da Hugging Face para carregar, treinar e usar modelos de NLP, como GPT, BERT e LLaMA.

einops: Facilita operações avançadas de manipulação de tensores, tornando-as mais intuitivas e eficientes, especialmente em deep learning.

accelerate: Otimiza o uso de hardware (CPU/GPU) para modelos de machine learning, facilitando o treinamento e a inferência distribuída.

bitsandbytes: Implementa técnicas de quantização para reduzir o tamanho e o consumo de memória de modelos grandes, permitindo execução em hardwares mais limitados.
"""


'\ntransformers: Biblioteca da Hugging Face para carregar, treinar e usar modelos de NLP, como GPT, BERT e LLaMA.\n\neinops: Facilita operações avançadas de manipulação de tensores, tornando-as mais intuitivas e eficientes, especialmente em deep learning.\n\naccelerate: Otimiza o uso de hardware (CPU/GPU) para modelos de machine learning, facilitando o treinamento e a inferência distribuída.\n\nbitsandbytes: Implementa técnicas de quantização para reduzir o tamanho e o consumo de memória de modelos grandes, permitindo execução em hardwares mais limitados.\n'

In [ ]:
!pip install -U --no-cache-dir \
  bitsandbytes==0.41.3 \
  accelerate==0.30.0 \
  transformers==4.40.0 \
  langchain \
  langchain-community \
  langchain-huggingface \
  langchainhub \
  langchain-chroma \
  einops==0.7.0 \
  pypdf pypdf2 \
  sentence-transformers

!pip install --upgrade transformers accelerate bitsandbytes


INFO: pip is looking at multiple versions of sentence-transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of sentence-transformers to determine which version is compatible with other requirements. This could take a while.
  Using cached transformers-4.54.1-py3-none-any.whl.metadata (41 kB)
  Using cached accelerate-1.9.0-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.54.1-py3-none-any.whl (11.2 MB)
Using cached accelerate-1.9.0-py3-none-any.whl (367 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: tokenizers
    Found existing ins

In [ ]:
import torch
import os
import getpass

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

from langchain.prompts import PromptTemplate
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
os.environ["HF_TOKEN"] = ""

In [ ]:
from huggingface_hub import login
login(token="")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quant_config
)

# 3. Criar pipeline Hugging Face para uso no LangChain
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.15,
    return_full_text=False
)

from langchain_community.llms import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device set to use cuda:0
/tmp/ipython-input-5-4274371303.py:35: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
template_rag = """
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Você é um assistente especializado na análise de questões da Olimpíada Cearense de Ciências Humanas (OCHE). Sua tarefa é classificar a dificuldade de cada questão como Fácil, Média ou Difícil, com base no contexto fornecido e na taxonomia revisada de Bloom.

Na OCHE, cada questão apresenta 4 itens. Três estão corretos em níveis diferentes, e o objetivo é identificar o item com maior pontuação. As questões incluem documentos de apoio, e o nível de dificuldade depende da complexidade da análise exigida.

Não responda a questão. Fundamente sua classificação com base no contexto e na taxonomia. Se não souber, apenas diga que não sabe.

Responda com:

Dificuldade: <nível>
Justificativa: <explicação curta, direta e fundamentada>
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Pergunta: {pergunta}
Contexto: {contexto}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""


In [ ]:
prompt_rag = PromptTemplate.from_template(template_rag)
print(prompt_rag)

input_variables=['contexto', 'pergunta'] input_types={} partial_variables={} template='\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente especializado na análise de questões da Olimpíada Cearense de Ciências Humanas (OCHE). Sua tarefa é classificar a dificuldade de cada questão como Fácil, Média ou Difícil, com base no contexto fornecido e na taxonomia revisada de Bloom.\n\nNa OCHE, cada questão apresenta 4 itens. Três estão corretos em níveis diferentes, e o objetivo é identificar o item com maior pontuação. As questões incluem documentos de apoio, e o nível de dificuldade depende da complexidade da análise exigida.\n\nNão responda a questão. Fundamente sua classificação com base no contexto e na taxonomia. Se não souber, apenas diga que não sabe.\n\nResponda com:\n\nDificuldade: <nível>\nJustificativa: <explicação curta, direta e fundamentada>\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nPergunta: {pergunta}\nContexto: {contexto}\n<|eot_i

In [ ]:
!pip install langchain
!pip install pypdf2
!pip install pypdf

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
def processar_pdf(path: str):
    loader = PyPDFLoader(path)
    docs = loader.load()

    # Substituir quebras de linha por espaço em cada página
    for doc in docs:
        doc.page_content = doc.page_content.replace("\n", " ")

    return docs


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# loader = WebBaseLoader(web_paths = ("https://www.bbc.com/portuguese/articles/cd19vexw0y1o",),)
# docs = loader.load()
# loader = PyPDFLoader(pdf_path)
# docs = loader.load()



pdf_path = "Críterios de Avaliação - RAG.pdf"
docs = processar_pdf(pdf_path)

In [ ]:
print(docs[0].page_content[:300])

Critérios  para  a  classificação  de  questões  da  OCHE  por   nível   de   dificuldade     Para  cumprir  o  objetivo  de  classificação  das  questões  da  olimpíada  cearense  de   ciências   humanas   (OCHE)   por   nível   de   dificuldade,   será   utilizado   um   conjunto   de   critérios 


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200, add_start_index = True)
splits = text_splitter.split_documents(docs)

In [ ]:
hf_embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = Chroma.from_documents(documents=splits, embedding=hf_embeddings)
retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs={"k": 1})

In [ ]:
prompt_rag = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template=template_rag,
)
prompt_rag

PromptTemplate(input_variables=['contexto', 'pergunta'], input_types={}, partial_variables={}, template='\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente especializado na análise de questões da Olimpíada Cearense de Ciências Humanas (OCHE). Sua tarefa é classificar a dificuldade de cada questão como Fácil, Média ou Difícil, com base no contexto fornecido e na taxonomia revisada de Bloom.\n\nNa OCHE, cada questão apresenta 4 itens. Três estão corretos em níveis diferentes, e o objetivo é identificar o item com maior pontuação. As questões incluem documentos de apoio, e o nível de dificuldade depende da complexidade da análise exigida.\n\nNão responda a questão. Fundamente sua classificação com base no contexto e na taxonomia. Se não souber, apenas diga que não sabe.\n\nResponda com:\n\nDificuldade: <nível>\nJustificativa: <explicação curta, direta e fundamentada>\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nPergunta: {pergunta}\nContexto: {

In [ ]:
def format_docs(docs, max_tokens=400):
    texto_total = ""
    token_count = 0

    for doc in docs:
        texto = doc.page_content.replace("\n", " ")
        tokens = len(texto.split())  # simples estimativa
        if token_count + tokens > max_tokens:
            break
        texto_total += texto + "\n\n"
        token_count += tokens

    return texto_total

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

rag_chain = (
    {"contexto": retriever | format_docs, "pergunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)


In [ ]:
def pdf_extract_text(pdf_path : str):
  loader = PyPDFLoader(pdf_path)
  text = loader.load()
  return text

def chunk_text(pdf_text):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
  chunks = text_splitter.split_documents(pdf_text)
  return chunks

from sentence_transformers import SentenceTransformer
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

current_dir = "/content/rag"
persistent_directory = os.path.join(current_dir, "db", "chroma_db_pdf")

def create_vector_store(chunks , db_path : str):
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
  db = Chroma.from_documents(documents = chunks, embedding = embedding_model, persist_directory=db_path)
  return db

def retriever_context(db : Chroma, query : str):
  retriever = db.as_retriever(search_type = "similarity", search_kwargs = {"k" : 3})
  relevant_chunks = retriever.invoke(query)

  return relevant_chunks

def build_context(relevant_chunks) -> str:
  context = ""
  for chunk in relevant_chunks:
    context += chunk.page_content
  return context

def build_chain(retriever, prompt):
    def rag_step(question):
        # Recupera contexto
        chunks = retriever.invoke(question)
        contexto = format_docs(chunks)

        # Formata o prompt com contexto e pergunta
        prompt_final = prompt.format(contexto=contexto, pergunta=question)

        # Passa para o LLM
        return llm.invoke(prompt_final)

    return rag_step




In [ ]:
import gc
import torch

ano = 2021
estrutura = """
Classifique a questão conforme a dificuldade, seguindo a taxonomia revisada de bloom:

Número da questão:
Dificuldade:
Justificativa:

você não deve responder a questão, deve atribuir a ela uma dificuldade.
"""

# Extrair e processar o texto da prova
prova_path = f"OCHE{ano}[2]FORMATADA.pdf"
prova = pdf_extract_text(prova_path)

texto_prova = ""
for i in range(len(prova)):
    texto_prova += prova[i].page_content

# Quebrar as questões
questions = texto_prova.strip().replace("\n", " ").replace("QUESTÃO", "OXENTE QUESTÃO").split("OXENTE")

try:
    questions.remove("")
except ValueError:
    pass
except:
    print("Erro ao remover questão vazia")
else:
    print("Questão vazia removida")


In [ ]:
questions.pop(0)
for i in questions:
  print(i)


 QUESTÃO  11    TEXTO  7    Em  1861,  a  condenação  das  irmãs  Ana  e  Quitéria  elucida  a  nulidade  de  defesa  das  mulheres  e  o  pouco  espaço   de   atenuantes   no   trato   do   infanticídio.   Entre   as   anotações   da   Chefatura   de   Polícia,   ficou   registrada   a   prisão   de   Antônio   Luiz   Pereira   e   suas   duas   filhas,   Ana   e   Quitéria   Pereira.   Moradores   de   Pacatuba,   os   três   foram   remetidos   à   Cadeia   Pública   de   Fortaleza.   Antônio   foi   preso   por   defloramento   de   suas   filhas.   Uma   delas   engravidou   –   a   nota   não   informou   qual   delas   –   e,   depois   do   parto,   as   moças   mataram   o   bebê   e   tentaram   ocultar   o   cadáver.   ( O   Cearense,   1861 ).   A   violência   cometida   pelo   pai   não   foi   suficiente   para   livrar   as   irmãs   da   pena   de   reclusão.   ( LIMA,   p.   13 )     Fonte:  LIMA,  A.  C.  P.  Da  maternidade  (re)negada:  mães  solteiras  e  mulheres

In [ ]:

# Construir a cadeia RAG
rag_chain = build_chain(retriever, template_rag)

# Abrir arquivo de saída
with open(f"OCHE{ano}[2] - Parâmetro 2.txt", "a") as arquivo:
    for n, question in enumerate(questions):
        # Montar prompt completo
        quest = estrutura + "\n\n" + question

        # Evitar rastreamento de gradientes e reduzir uso de memória
        with torch.no_grad():
            resposta = rag_chain(quest)
            for i in range(10):
                gc.collect()
                torch.cuda.empty_cache()


        # Exibir e salvar resposta
        print(f"Número da questão: {n+11}")
        print(resposta)
        arquivo.write(f"\nNúmero da questão: {n+11}\n{resposta}\n\n\n")

        # 🔥 Liberar memória (RAM + GPU)
        del resposta, quest
        for i in range(5):
          gc.collect()
          torch.cuda.empty_cache()

# 📦 Download do arquivo final
from google.colab import files
files.download(f"OCHE{ano}[2] - Parâmetro 2.txt")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Número da questão: 11
Dificuldade: Difícil
Justificativa: A questão exige uma análise crítica e profunda sobre o tema de infanticídio no Ceará do século XIX, incluindo a dinâmica social, a percepção da sociedade sobre a gravidez fora do casamento e a construção histórica do papel da mulher enquanto mãe abnegada. Além disso, a questão requer a compreensão dos mecanismos legais e sociais que influenciaram a punição das mulheres envolvidas no crime, bem como a capacidade de avaliar a perspectiva da autora sobre o assunto. Essas habilidades são características do nível de "Análise" da taxonomia de Bloom.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Número da questão: 12
Dificuldade: Difícil

Justificativa: A questão requer do participante uma análise profunda e interpretativa do texto, além de uma busca por informações adicionais fora do documento fornecido. A resposta correta depende de uma compreensão matizada do conceito de violência e suas implicações filosóficas, além de uma capacidade de distinguir entre as afirmações de autores como Jean-Paul Sartre, Hannah Arendt e Frantz Fanon. Isso demonstra uma abordagem de alta complexidade, exigindo habilidades de pensamento crítico, análise e interpretação.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Número da questão: 19
Dificuldade: D
Justificativa: Devido à presença de várias dimensões narrativas complexas, como ironia, metalinguagem, metonímia, referências culturais e símbolos, bem como a interconexão entre temas sociais, psicológicos e existenciais, essa crônica requer uma análise profunda e contextualizada, movendo-se entre diferentes categorias cognitivas, incluindo compreensão, interpretação, reflexão e criação. Além disso, a crônica parece fazer parte de uma obra literária contemporânea de um autor acadêmico, sugerindo a inclusão de elementos neoparnasianos e linguística.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Número da questão: 20
Dificuldade: Difícil
Justificativa: A questão requer a aplicação de categorias cognitivas de ordem superior, como analisar e interpretar textos complexos, reconhecer paralelos históricos entre figuras literárias e filosóficas, e fazer conexões entre ideias de diferentes contextos. Além disso, a questão exige a capacidade de pensar críticamente e contextualizar as informações fornecidas, o que é característico de categorias cognitivas de ordem superior.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
for question in questions:
        # Montar prompt completo
        quest = estrutura + "\n\n" + question
        resposta = rag_chain(quest)
        print("\n🧠 RESPOSTA DO MODELO:\n")
        print(resposta)


In [ ]:
for i in range(100):
  import gc
  import torch
  gc.collect()
  torch.cuda.empty_cache()